# Semana 1: Introducción a Scala y Programación Funcional

## Contexto del Ejercicio: Mi Porfolio como Data Engineer

Este notebook forma parte de la actividad **"Mi porfolio como Data Engineer"** descrita en el syllabus (páginas 18-19). 

El objetivo es que **construyas tu propia base de conocimiento** documentando lo aprendido. Puedes utilizar estos notebooks como:
*   **Guía**: Para seguir los conceptos clave.
*   **Base**: Para ampliar con tus propios ejemplos y notas.
*   **Complemento**: A tu repositorio de código en GitHub.

Lo importante es que al final del módulo tengas un **recurso propio** que demuestre tu comprensión de Scala y la programación funcional.

---

Este notebook sirve como guía introductoria para el desarrollo del portfolio de Data Engineering. Cubriremos los conceptos básicos de la programación funcional y el lenguaje Scala, preparando el terreno para trabajar con Apache Spark.

## 1. Limitaciones del Entorno y Resolución de Problemas (Troubleshooting)

Al trabajar con notebooks de Scala usando el kernel **Almond** (basado en **Ammonite REPL**), existen algunas limitaciones que debes conocer para evitar frustraciones:

### 1.1 El error de "Compilation Failed" con `var` e `import`
A veces, al redefinir una variable `var` o al ejecutar celdas de forma desordenada, el intérprete puede devolver un error críptico como:
`cmdX.sc:YY: ')' expected but 'import' found. import variable$value.{value => variable}`

*   **Causa**: El REPL de Ammonite envuelve cada celda en un objeto interno. A veces, el rastreo de estados de variables mutables se corrompe en el envoltorio.
*   **Solución**: **Reiniciar el Kernel** (Menú *Kernel* -> *Restart*). Esto limpia el estado y permite volver a compilar sin errores.

### 1.2 Redefinición de `val` y `def`
A diferencia de un archivo `.scala` estándar donde no puedes tener dos `val` con el mismo nombre en el mismo ámbito, en el notebook puedes redefinirlos en celdas distintas. La celda ejecutada más recientemente "machaca" a la anterior.

### 1.3 Carga de librerías con `$ivy` 
Para añadir dependencias externas usamos la sintaxis `$ivy`. 
*   **Regla**: Debe ir en una celda **sola** o al principio de los imports. Si falla, el reinicio de kernel suele ser obligatorio.
```scala
import $ivy.`org.typelevel::cats-core:2.9.0`
```

## 2. Conceptos Teóricos: Fundamentos de Scala

Antes de empezar a programar, es importante asentar los conceptos. Responde brevemente a las siguientes cuestiones (puedes editar esta celda de Markdown):

**Q1. ¿Por qué crees que seleccionamos Scala como lenguaje para el ecosistema Spark en lugar de usar solo Python o Java?**

Scala fue elegido como lenguaje base de Spark porque ofrece integración nativa con la JVM, permitiendo máximo rendimiento y acceso directo a las optimizaciones internas del motor. Al ser un lenguaje funcional con tipado estático fuerte, promueve inmutabilidad y transformaciones puras, fundamentales en sistemas distribuidos tolerantes a fallos.

Python sigue siendo ideal para prototipado y ML por su ecosistema, y Java para integraciones enterprise, pero Scala ofrece el mejor equilibrio entre rendimiento, expresividad y compatibilidad con Spark.

**Q2. Explica con tus palabras la diferencia entre `val` y `var`. ¿Cuál deberíamos priorizar en un entorno de Big Data?**

**val:** crea un binding inmutable (no se puede reasignar)
**var:** permite reasignación (estado mutable)

En Big Data deberíamos priorizar val porque la inmutabilidad facilita concurrencia, reproducibilidad y razonamiento sobre el código distribuido, reduce errores por condiciones de carrera y encaja con el paradigma funcional de Spark

**Q3. ¿Qué significa que una función sea "pura" y por qué esto ayuda a procesar datos en un cluster de máquinas?**

Una función pura es determinística y libre de efectos secundarios. Esto significa que su ejecución depende únicamente de sus parámetros de entrada y no altera estado compartido.

En entornos distribuidos como Spark, esta propiedad permite paralelización segura, ejecución idempotente y recomputación basada en lineage. Gracias a esto, el sistema puede re-ejecutar tareas fallidas en distintos nodos sin comprometer consistencia ni requerir mecanismos complejos de coordinación.



## 3. Introducción a la Programación Funcional

La programación funcional (FP) es un paradigma de programación donde los programas se construyen aplicando y componiendo funciones. 

### Conceptos Clave
*   **Inmutabilidad**: Los datos no cambian una vez creados. En lugar de modificar una variable, creamos una nueva.
*   **Funciones Puras**: El resultado de una función depende solo de sus argumentos y no tiene efectos secundarios (como imprimir en consola o modificar variables globales).
*   **Funciones de Orden Superior**: Funciones que pueden tomar otras funciones como argumentos o devolverlas como resultados.

### Ventajas
*   **Facilidad para el paralelismo**: Al no haber estado compartido mutable, es más seguro ejecutar código en paralelo (crucial para Spark).
*   **Código más predecible y testeable**: Las funciones puras son deterministas.
*   **Modularidad**: El código se compone de pequeñas funciones reutilizables.

## 4. Introducción a Scala

Scala (Scalable Language) combina la programación orientada a objetos y la programación funcional. Es el lenguaje en el que está escrito Spark.

### Sintaxis Básica y Tipos

In [ ]:
// Variables inmutables (val) vs mutables (var)
val mensajeInmutable = "Hola, esto no puede cambiar"
// mensajeInmutable = "Nuevo valor" // Esto daría error

var mensajeMutable = "Hola, esto sí puede cambiar"
mensajeMutable = "Nuevo valor"
println(mensajeMutable)

In [2]:
// Tipos de datos básicos
val numero: Int = 42
val decimal: Double = 3.14
val booleano: Boolean = true
val texto: String = "Scala es genial"

println(s"Texto: $texto, Número: $numero") // Interpolación de cadenas con s"..."

Texto: Scala es genial, Número: 42


numero: Int = 42
decimal: Double = 3.14
booleano: Boolean = true
texto: String = "Scala es genial"

### Funciones

En Scala, las funciones son ciudadanos de primera clase.

In [1]:
// Definición básica de una función
def suma(a: Int, b: Int): Int = {
  a + b
}

println(s"Suma: ${suma(5, 3)}")

Suma: 8


defined function suma

## 5. Ejemplo de Programación Funcional en Scala

Vamos a ver cómo manipular colecciones usando funciones de orden superior como `map`, `filter` y `reduce`. Esto es la base de cómo manipularemos datos en Spark.

In [4]:
val numeros = List(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)

// 1. Filter: Filtrar números pares
val pares = numeros.filter(n => n % 2 == 0)
println(s"Pares: $pares")

// 2. Map: Multiplicar cada número por 2
val doblados = pares.map(n => n * 2)
println(s"Doblados: $doblados")

// 3. Reduce: Sumar todos los elementos
val sumaTotal = doblados.reduce((a, b) => a + b)
println(s"Suma Total: $sumaTotal")

// Encademaniento de operaciones (Pipeline)
val resultado = numeros
  .filter(_ % 2 == 0) // Sintaxis concisa usando _
  .map(_ * 2)
  .sum // sum es una reducción especializada

println(s"Resultado encadenado: $resultado")

Pares: List(2, 4, 6, 8, 10)
Doblados: List(4, 8, 12, 16, 20)
Suma Total: 60
Resultado encadenado: 60


numeros: List[Int] = List(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
pares: List[Int] = List(2, 4, 6, 8, 10)
doblados: List[Int] = List(4, 8, 12, 16, 20)
sumaTotal: Int = 60
resultado: Int = 60

## 6. Ejercicios Prácticos: Nivel Básico

Estos ejercicios te ayudarán a familiarizarte con la sintaxis de Scala.

### Ejercicio 1: Variables y Cadenas
Crea una variable inmutable que guarde tu saludo favorito y una variable mutable que guarde un contador. Luego imprime ambos. Incrementa el contador e imprime de nuevo.

In [6]:
// TODO: Tu código aquí
 val saludo = "Hola, mundo!"
var contador = 1

println(saludo)
println(s"Contador: $contador")
contador += 1
println(s"Contador actualizado: $contador")

Hola, mundo!
Contador: 1
Contador actualizado: 2


saludo: String = "Hola, mundo!"
contador: Int = 2

### Ejercicio 2: Funciones Simples
Define una función llamada `saludar` que reciba un nombre (String) y devuelva un String con el texto "¡Bienvenido al Máster, [nombre]!". Pruébala con tu nombre.

In [9]:
// TODO: Tu código aquí
def saludar(nombre: String): String = {
    s"Hola, $nombre!"
}   

defined function saludar

## 7. Ejercicios con Colecciones (Listas y Mapas)

Las colecciones son esenciales en Scala y Spark. Recuerda que por defecto son inmutables.

### Ejercicio 3: Manipulación de Listas
Dada una lista de ciudades, realiza las siguientes operaciones:
1. Filtra las ciudades que tienen más de 5 letras.
2. Convierte todas las ciudades a minúsculas.
3. Cuenta cuántas ciudades cumplen la condición.


In [11]:
val ciudades = List("Madrid", "Barcelona", "Valencia", "Sevilla", "Bilbao")

// TODO: Tu código aquí
val resultado = ciudades
  .filter(_.length > 6) // Filtrar ciudades con nombre mayor a 6 caracteres
  .map(_.toUpperCase) // Convertir a mayúsculas
  .sorted // Ordenar alfabéticamente

ciudades: List[String] = List(
  "Madrid",
  "Barcelona",
  "Valencia",
  "Sevilla",
  "Bilbao"
)
resultado: List[String] = List("BARCELONA", "SEVILLA", "VALENCIA")

### Ejercicio 4: Uso de Mapas (Diccionarios)
Crea un mapa que relacione nombres de productos con su precio. Luego calcula el precio de un "pack" que contenga dos productos distintos sumando sus valores desde el mapa.

In [12]:
// TODO: Crear mapa
val precios = Map("Procesador" -> 300, "Memoria" -> 150, "Disco" -> 100)

// TODO: Sumar precios
val totalPack = precios.values.sum

precios: Map[String, Int] = Map(
  "Procesador" -> 300,
  "Memoria" -> 150,
  "Disco" -> 100
)
totalPack: Int = 550

## 8. Ejercicios Prácticos: Nivel Intermedio

Usa estos ejercicios como base para practicar o como ejemplos, y documentar en tu porfolio.

### Ejercicio 5: Estructuras de Control (Match)
Implementa una función que clasifique una edad en etapas de la vida usando un `match` con guardas:
*   < 13: "Niño"
*   13-17: "Adolescente"
*   18-64: "Adulto"
*   >= 65: "Senior"

In [13]:
def clasificarEdad(edad: Int): String = {
 
  edad match {
    case e if e >= 0 && e <= 12 => "Niño"
    case e if e >= 13 && e <= 19 => "Adolescente"
    case e if e >= 20 && e <= 64 => "Adulto"
    case e if e >= 65 => "Anciano"
    case _ => "Edad no válida"
  }
} 

println(clasificarEdad(25)) // Debería imprimir "Adulto"

Adulto


defined function clasificarEdad

### Ejercicio 6: Funciones y Currying
Crea una función currificada para validar si una cadena tiene una longitud mínima. Luego crea una función específica llamada `validarMin5` que use la primera con el valor 5 fijado.

In [14]:
// 1. Función currificada
def validarLongitud(min: Int)(texto: String): Boolean = {
  texto.length >= min
}

// 2. Función parcialmente aplicada
val validarMin5 = validarLongitud(5) _ 

validarMin5("Hola") // Debería ser false

defined function validarLongitud
validarMin5: String => Boolean = ammonite.$sess.cmd14$Helper$$Lambda$2783/0x000000780197ee30@1d25b0f2
res14_2: Boolean = false

### Ejercicio 7: For Comprehension
Dada una lista de nombres, obtén una lista que contenga solo los nombres que empiezan por 'A' y conviértelos a mayúsculas usando un `for yield`.

In [16]:
val nombres = List("Alice", "Bob", "Charlie", "Anna", "David")

val nombresConA = for { 
  n <- nombres if n.startsWith("A")
} yield n.toUpperCase

println(nombresConA)

List(ALICE, ANNA)


nombres: List[String] = List("Alice", "Bob", "Charlie", "Anna", "David")
nombresConA: List[String] = List("ALICE", "ANNA")